In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:

# Task 1: Write your code here:

# 1.Load dataset (USE THE PATH PRINTED BY kagglehub)
df = pd.read_csv(f"{path}/Q1_data.csv")


In [ ]:
# Task 2: Write your code here:

# 2. Inspect first rows
print(df.head())

In [ ]:
# Task 3: Write your code here:

# 3. Dataset info
df.info()

In [ ]:
# Task 4: Write your code here:

# 4. Statistical description
print(df.describe())

In [ ]:
# Task 5: Write your code here:

# 5.Plot the target distribution (delivery_time)
def check_target_distribution(df, target_column):
    plt.figure(figsize=(10, 5))
    df[target_column].hist(bins=30, edgecolor='red')
    plt.title("Delivery Time Distribution")
    plt.xlabel("Delivery Time")
    plt.ylabel("Frequency")
    plt.grid(False)
    plt.show()
df = df.rename(columns={"Delivery_Time": "delivery_time"})
check_target_distribution(df, "delivery_time")


In [ ]:
# Task 1: Write your code here:

if "order_id" in df.columns.str.lower():
    df = df.drop(columns=[col for col in df.columns if col.lower() == "order_id"])


In [ ]:
# Task 2: Write your code here:
# Check missing values
print(df.isnull().sum())

for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ["int64", "float64"]:
            df[col].fillna(df[col].median(), inplace=True)
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)


In [ ]:
# Task 3: Write your code here:
# Check and remove duplicates
print("Duplicates before:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicates after:", df.duplicated().sum())


In [ ]:
# Task 4: Write your code here:
# Encode categorical variables
from sklearn.preprocessing import OneHotEncoder

categorical_cols = df.select_dtypes(include=["object"]).columns
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)


In [ ]:
# Task 5: Write your code here:
# Apply feature scaling for all features
from sklearn.preprocessing import StandardScaler #StandardScaler

scaler = StandardScaler()

X = df.drop(columns=["delivery_time"])   # target out
y = df["delivery_time"]

X_scaled = scaler.fit_transform(X)


In [ ]:
# Task 6: Write your code here:

# Check for target imbalance and state if it is imbalanced or not
y.value_counts(normalize=True)


In [ ]:
# Task 1: Write your code here:
# 1. Split the dataset into features (X) and target (y)

from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

X = df.drop(columns=["delivery_time"])
y = df["delivery_time"]



In [ ]:
# Task 2,3,4,5: Write your code here:

kf = KFold(n_splits=5, shuffle=True, random_state=42) # KFold

mae_scores = []
 # tarin the model
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)


    #MAE
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

#printing the Average MAE across folds
print("Average MAE across folds:", np.mean(mae_scores))



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Features and target
X = df.drop("delivery_time", axis=1)
y = df["delivery_time"]

# Column types
categorical_cols = X.select_dtypes(include="object").columns
numerical_cols = X.select_dtypes(exclude="object").columns

# Preprocessing
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numerical_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols)
])

# Pipeline with RandomForest
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

# Fit the model
pipeline.fit(X, y)

# Feature Importance Plot
# Get transformed feature names directly from preprocessor
all_features = pipeline.named_steps['preprocessor'].get_feature_names_out()

# Get feature importances from the RandomForest model
importances = pipeline.named_steps['model'].feature_importances_
indices = np.argsort(importances)[::-1]  # sort descending

# Plot
plt.figure(figsize=(12, 6))
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), all_features[indices], rotation=90)
plt.title("Feature Importance")
plt.show()




In [ ]:
# Task 2: Write your code here:

# Predicted Delivery Time Histogram
y_pred = pipeline.predict(X)

plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=30, edgecolor='black')
plt.title("Predicted Delivery Time Distribution")
plt.xlabel("Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.show()


In [ ]:
# Task Bonus: Write your code here:


from IPython.display import clear_output
%pip install kagglehub catboost lightgbm tqdm -q
clear_output()

import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Features and target
X = df.drop("delivery_time", axis=1)
y = df["delivery_time"]

categorical_cols = X.select_dtypes(include="object").columns
numerical_cols = X.select_dtypes(exclude="object").columns

# Preprocessing pipeline
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numerical_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols)
])

# Define KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

for train_idx, val_idx in kf.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # RandomForest pipeline

    rf_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
    ])
    rf_pipeline.fit(X_train, y_train)
    rf_pred = rf_pipeline.predict(X_val)

    # CatBoost pipeline
    cb_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", CatBoostRegressor(
            iterations=500,
            learning_rate=0.1,
            depth=6,
            verbose=0,
            random_state=42
        ))
    ])
    cb_pipeline.fit(X_train, y_train)
    cb_pred = cb_pipeline.predict(X_val)

    # Average predictions

    avg_pred = (rf_pred + cb_pred) / 2

    # MAE for this fold
    mae = mean_absolute_error(y_val, avg_pred)
    mae_scores.append(mae)

# Final result
print(f"MAE per fold: {mae_scores}")
print(f"Average MAE across folds: {np.mean(mae_scores):.2f}")
